# Bar Path Model Training

Training an model for bar path tracking using exercise data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from os import path
import pickle as pkl

### Load data and config

In [3]:
from utils import load_data, load_config

data = load_data()
config = load_config()
data.head()

Fetching data from Airtable...
Formula: AND({ValidData} = 'true', {StartRepTime} != '', {EndRepTime} != '', {Reps} >= '1')
Found 511 matches


,Date,Exercise,Lifter,WorkoutTime,Reps,Weight,Intensity,Notes,StartRepTime,EndRepTime,Counter,TimeBetweenSamples,AccX,AccY,AccZ,Pitch,Roll,Yaw,HeartRate
0,"Jul 17, 2024 at 8:37:11 PM",DB Side Raise,Anwar,32.25,9,27.5,9,None,4.38,26.48,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[663011, 9, 9, 10, 10, 10, 13, 7, 10, 10, 10, ...","[-0.327, 0.068, 0.159, -0.036, -0.125, -0.051,...","[-0.15, -0.236, -0.011, 0.169, 0.05, -0.098, -...","[0.03, 0.009, -0.077, -0.052, 0.05, 0.151, 0.1...","[0.346, 0.346, 0.347, 0.347, 0.348, 0.348, 0.3...","[0.093, 0.093, 0.092, 0.092, 0.092, 0.092, 0.0...","[-0.016, -0.016, -0.015, -0.015, -0.015, -0.01...","[110, 110, 110, 110, 110, 110, 110, 110, 110, ..."
1,"May 20, 2024 at 1:44:48 PM",Bench,Anwar,47.67,7,195,10,None,21.12,40.38,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[258264, 7, 9, 9, 10, 10, 10, 10, 11, 9, 10, 1...","[0.13, -0.052, -0.069, -0.006, -0.048, 0.075, ...","[-0.17, -0.157, -0.054, -0.044, -0.078, -0.024...","[0.147, 0.036, -0.106, -0.231, -0.274, -0.156,...","[0.15, 0.15, 0.151, 0.151, 0.152, 0.152, 0.152...","[0.059, 0.059, 0.058, 0.058, 0.058, 0.058, 0.0...","[-0.004, -0.004, -0.003, -0.002, -0.002, -0.00...","[93, 93, 93, 93, 93, 93, 93, 93, 93, 93, 93, 9..."
2,"Apr 16, 2024 at 4:45:01 PM",Horizontal Chest Press,Anwar,28.69,8,145,10,None,6.72,25.34,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[222003, 10, 9, 10, 10, 10, 12, 7, 10, 10, 10,...","[0.254, 0.409, -0.13, -0.019, 0.234, 0.331, 0....","[0.819, -0.072, -0.617, -1.169, -0.67, 0.128, ...","[0.095, 0.315, 0.715, 0.66, 0.735, 0.698, 0.82...","[0.667, 0.674, 0.676, 0.683, 0.694, 0.704, 0.7...","[0.057, 0.059, 0.063, 0.068, 0.074, 0.078, 0.0...","[-0.02, -0.022, -0.027, -0.034, -0.038, -0.04,...","[84, 84, 84, 84, 84, 84, 84, 84, 84, 84, 84, 8..."
3,"Apr 27, 2024 at 11:07:05 AM",Squat,Anwar,46.75,7,225,9,None,16.08,39.98,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[242417, 13, 7, 9, 11, 9, 10, 10, 10, 12, 8, 1...","[0.119, -0.129, -0.155, -0.134, -0.003, 0.076,...","[0.138, -0.047, -0.156, -0.166, -0.181, -0.138...","[0.152, 0.149, 0.097, 0.144, 0.222, 0.199, 0.0...","[0.526, 0.528, 0.529, 0.529, 0.528, 0.527, 0.5...","[0.023, 0.022, 0.022, 0.022, 0.022, 0.022, 0.0...","[-0.006, -0.005, -0.005, -0.005, -0.004, -0.00...","[124, 124, 124, 124, 124, 124, 124, 124, 124, ..."
4,"Mar 2, 2024 at 3:35:29 PM",DB Side Raise,Anwar,32.91,12,27.5,8,None,4.05,30.96,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[95273, 10, 11, 11, 8, 11, 10, 9, 11, 9, 10, 1...","[-0.151, 0.003, -0.038, 0.02, 0.008, -0.123, -...","[0.203, 0.355, 0.324, 0.137, 0.029, -0.009, 0....","[-0.006, -0.054, -0.154, -0.193, -0.148, -0.10...","[0.401, 0.402, 0.403, 0.404, 0.405, 0.405, 0.4...","[0.123, 0.124, 0.124, 0.124, 0.124, 0.123, 0.1...","[-0.025, -0.025, -0.024, -0.023, -0.023, -0.02...","[123, 123, 123, 123, 123, 123, 123, 123, 123, ..."


## Train an exercise classifier

For each data column

- Extract features from the entire time window for each data set
- Train model to predict exercise based on these feature


#### To Do

- Add back FFT features
- Feature selection
- Feature normalization
- Plot classifier results

### Extract features and labels

In [4]:
from preprocessing import preprocess_data, extract_features, extract_labels

data = preprocess_data(data, config['CutoffFreq'], config['SampleRate'])
features = extract_features(data)
labels = extract_labels(data)

### Save features and labels

In [5]:
features_file = path.join(config['ExercisesFolder'], config['FeaturesFilename'])
labels_file = path.join(config['ExercisesFolder'], config['LabelsFilename'])

features.to_csv(features_file, index=False)
labels.to_csv(labels_file, index=False)

### Train model

In [6]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2)

clf = XGBClassifier()
clf.fit(X_train, y_train)

print("Training accuracy: ", clf.score(X_train, y_train))
print("Testing accuracy: ", clf.score(X_test, y_test))

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))

Training accuracy:  1.0
Testing accuracy:  0.9611650485436893
              precision    recall  f1-score   support

           1       0.96      1.00      0.98        25
           2       0.67      0.67      0.67         3
           3       1.00      0.67      0.80         3
           5       1.00      1.00      1.00         3
           6       0.00      0.00      0.00         0
           7       1.00      1.00      1.00        10
           8       1.00      1.00      1.00         4
          10       0.90      1.00      0.95         9
          11       1.00      1.00      1.00         1
          12       1.00      1.00      1.00        21
          13       1.00      1.00      1.00        12
          14       0.00      0.00      0.00         1
          15       1.00      0.90      0.95        10
          16       1.00      1.00      1.00         1

    accuracy                           0.96       103
   macro avg       0.82      0.80      0.81       103
weighted avg      

### Save model

In [7]:
classifier_file = path.join(config['ExercisesFolder'], config['ClassifierFilename'])
with open(classifier_file, 'wb') as f:
    pkl.dump(clf, f)

## Train a predictor for whether is in rep state or non rep state

For entries with a RepStartTime and RepEndTime
 - Extract windows for each axes
 - Label window with 0 or 1 depending on whether it is between RepStartTime and RepEndTime
 - Create a classifier for this data

#### To Do

- Add back in FFT features
- Feature selection
- Feature normalization
- Plot classifier results

### Extract windowed features and labels

In [8]:
from preprocessing import extract_window_features, extract_window_labels

config = load_config()
window_features = extract_window_features(data, config['WindowLength'], config['WindowStride'])
window_labels = extract_window_labels(data, config['WindowLength'], config['WindowStride'], config['SampleRate'])

### Save windowed features and labels

In [9]:
features_file = path.join(config['RepsFolder'], config['FeaturesFilename'])
labels_file = path.join(config['RepsFolder'], config['LabelsFilename'])

window_features.to_csv(features_file, index=False)
window_labels.to_csv(labels_file, index=False)

### Train model

In [10]:
X_train, X_test, y_train, y_test = train_test_split(window_features, window_labels, test_size=0.2)

clf = XGBClassifier()

clf.fit(X_train, y_train)
print("Training accuracy: ", clf.score(X_train, y_train))
print("Testing accuracy: ", clf.score(X_test, y_test))

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))

Training accuracy:  0.9998220260553855
Testing accuracy:  0.9605637813211845
              precision    recall  f1-score   support

           0       0.96      0.95      0.96      3159
           1       0.96      0.97      0.96      3865

    accuracy                           0.96      7024
   macro avg       0.96      0.96      0.96      7024
weighted avg       0.96      0.96      0.96      7024



### Save model

In [11]:
classifier_file = path.join(config['RepsFolder'], config['ClassifierFilename'])
with open(classifier_file, 'wb') as f:
    pkl.dump(clf, f)